In [ ]:
# ============================================================
# Model 3: Gated Fusion
# ============================================================
# 텍스트 인코더 (BERT [CLS]) + 이미지 인코더 (EfficientNet pooling)
# → 학습 가능한 게이트(sigmoid)가 모달리티별 기여도를 동적 조절
# fused = gate * text + (1-gate) * image
# 장점: 샘플별 모달리티 중요도 자동 학습 / 단점: 게이트 편향 가능성
# ============================================================

from common import *

set_seed(42)
device = get_device()

# --- 데이터 설정 (필요시 수정) ---
DATA_CONFIG = DEFAULT_DATA_CONFIG.copy()
# DATA_CONFIG['spectrogram_type'] = 'mel'

df = load_data(DATA_CONFIG)
visualize_data(df)
train_ds, val_ds, test_ds, class_weights, tokenizer = prepare_datasets(df, DATA_CONFIG, device)

In [ ]:
# ============================================================
# 모델 아키텍처
# ============================================================

class GatedFusionModel(nn.Module):
    def __init__(self, text_model_name, image_model_name,
                 text_hidden=768, image_hidden=1280,
                 fusion_hidden=256, dropout=0.3, num_classes=4,
                 freeze_text=False, freeze_image=False):
        super().__init__()

        self.text_encoder = AutoModel.from_pretrained(text_model_name)
        self.image_encoder = timm.create_model(
            image_model_name, pretrained=True, num_classes=0)

        if freeze_text:
            for p in self.text_encoder.parameters():
                p.requires_grad = False
        if freeze_image:
            for p in self.image_encoder.parameters():
                p.requires_grad = False

        self.text_proj = nn.Sequential(
            nn.Linear(text_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout))

        self.image_proj = nn.Sequential(
            nn.Linear(image_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout))

        # Gating: [text, image] → sigmoid gate
        self.gate_network = nn.Sequential(
            nn.Linear(fusion_hidden * 2, fusion_hidden),
            nn.ReLU(),
            nn.Linear(fusion_hidden, fusion_hidden),
            nn.Sigmoid())  # 0~1

        self.classifier = nn.Sequential(
            nn.Linear(fusion_hidden, fusion_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(fusion_hidden, num_classes))

    def forward(self, input_ids, attention_mask, images):
        text_out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        text_feat = self.text_proj(text_out.last_hidden_state[:, 0, :])  # [CLS]

        image_feat = self.image_proj(self.image_encoder(images))  # GAP

        combined = torch.cat([text_feat, image_feat], dim=1)
        gate = self.gate_network(combined)  # [B, D]

        # gate * text + (1-gate) * image
        fused = gate * text_feat + (1 - gate) * image_feat
        return self.classifier(fused)

In [ ]:
# ============================================================
# 모델 & 학습 설정 (수정 가능)
# ============================================================

MODEL_CONFIG = {
    'text_model_name': DATA_CONFIG['text_model_name'],
    'image_model_name': DATA_CONFIG['image_model_name'],
    'text_hidden': 768,
    'image_hidden': 1280,
    'fusion_hidden': 256,
    'dropout': 0.4,              # 0.1~0.5
    'num_classes': 4,
    'freeze_text': False,        # True → 인코더 동결 (빠른 학습)
    'freeze_image': False,
}

TRAIN_CONFIG = {
    'batch_size': 16,            # GPU 메모리에 따라 8/16/32
    'encoder_lr': 1e-5,          # 사전학습 인코더용 (1e-5~5e-5)
    'classifier_lr': 1e-3,       # 새 레이어용 (1e-4~1e-2)
    'weight_decay': 0.02,
    'num_epochs': 15,
    'patience': 5,               # Early stopping
    'scheduler': 'cosine',       # 'cosine' | 'plateau' | 'warmup_cosine'
    'warmup_steps': 50,          # warmup_cosine 전용
    'max_grad_norm': 1.0,
}

In [ ]:
# ============================================================
# 학습 & 평가
# ============================================================

set_seed(DATA_CONFIG['random_seed'])

model = GatedFusionModel(**MODEL_CONFIG)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total_p:,}  Trainable: {train_p:,}')

result = train_and_evaluate(
    model, train_ds, val_ds, test_ds,
    TRAIN_CONFIG, class_weights, device)

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [ ]:
# ============================================================
# 결과 시각화
# ============================================================

plot_training_curves(result, 'Model 3: Gated Fusion')
plot_confusion(result, 'Model 3: Gated Fusion')
print_report(result, 'Model 3: Gated Fusion')